In [1]:
#!/usr/bin/env python3
# train_FG_long_k_fixed.py
# Fixed: removed undefined k_map usage and removed printing of model parameter count.
# Convert wide (ETA, F1..F12, G1..G12) -> long dataset with mapping k->(Ri,M)
# Train NN to predict (Fval,Gval) from inputs (Ri, M, Eta, k).
# Optional: you can later extend pde_residuals(...) and set USE_PDE=True for PINN.

import os, time, json
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split

# ---------------- User settings ----------------
DATA_PATH = "Grad-4b-Merged.csv"   # your CSV
OUT_DIR = "Grad-4b-pinn-results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# mapping k -> (Epsilon, N) as you described
Cf_map = {
    1: (0, 20), 2: (0.1, 20), 3: (0.3, 20), 4: (0.5, 20)
}

"Extract Data.ipynb"
# training hyperparams (tune)
SEED = 42
N_EPOCHS = 12000
BATCH_SIZE = 128
LR = 2e-3
ONECYCLE_MAX_LR = 8e-3
GRAD_CLIP = 5.0
LBFGS_ITERS = 200
EARLY_STOPPING_PATIENCE = 3000
PRINT_EVERY = 100

USE_PDE = True   # set True only after you implement pde_residuals()
# ------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)

In [2]:
# ---------------- Load CSV and build long dataset ----------------
df_w = pd.read_csv(DATA_PATH)
print("Loaded CSV:", DATA_PATH, "shape:", df_w.shape)
print("Columns:", df_w.columns.tolist()[:20])

# find ETA column
eta_col = next((c for c in df_w.columns if c.strip().lower()=='eta' or 'eta' in c.lower()), None)
if eta_col is None:
    raise RuntimeError("ETA column not found in CSV.")

# check F/G columns exist
Cf_cols = [f"Cf{i}" for i in range(1,5)]

for c in Cf_cols: #+G_cols:
    if c not in df_w.columns:
        raise RuntimeError(f"Column {c} not found in CSV.")

# Build long records: for each k create rows with assigned Ri,M
records = []
n_eta = df_w.shape[0]
for k in range(1,5):
    epsilon, n  = Cf_map[k]
    Cf_col = f"Cf{k}" #; Gcol = f"G{k}"
    for i in range(n_eta):
        records.append({
            "Epsilon": epsilon,
            "N": n,
            "Eta": float(df_w.iloc[i][eta_col]),
            "k": float(k),  # kept only for grouping, NOT used as model input,           # include k as numeric (we'll normalize)
            "Cf_val": float(df_w.iloc[i][Cf_col])
        })
df = pd.DataFrame.from_records(records)
print("Built long dataframe shape:", df.shape)
df.to_csv(os.path.join(OUT_DIR,"Cf_long_raw.csv"), index=False)

Loaded CSV: Grad-4b-Merged.csv shape: (252, 5)
Columns: ['ETA', 'Cf1', 'Cf2', 'Cf3', 'Cf4']
Built long dataframe shape: (1008, 5)


In [3]:
# ---------------- Prepare model inputs/outputs ----------------
# Inputs: [Epsilon, N, Eta]  (k removed from model inputs) -> Outputs: [Cf_val]
X = df[["Epsilon","N","Eta"]].values.astype(np.float32)
Y = df[["Cf_val"]].values.astype(np.float32)

# Normalize inputs per-column (important)
X_mean = X.mean(axis=0); X_std = X.std(axis=0) + 1e-12
Y_mean = Y.mean(axis=0); Y_std = Y.std(axis=0) + 1e-12
Xn = (X - X_mean)/X_std
Yn = (Y - Y_mean)/Y_std

# Train/val/test split (stratify by (Epsilon,N) combos so each combo appears)
df["combo"] = df["Epsilon"].astype(str) + "_" + df["N"].astype(str)
idx = np.arange(Xn.shape[0])
train_idx, test_idx = train_test_split(idx, test_size=0.18, random_state=SEED, stratify=df['combo'])
# from test split further into val/test
val_idx, test_idx = train_test_split(test_idx, test_size=0.5, random_state=SEED, stratify=df['combo'].iloc[test_idx])

X_train = Xn[train_idx]; Y_train = Yn[train_idx]
X_val   = Xn[val_idx];   Y_val   = Yn[val_idx]
X_test  = Xn[test_idx];  Y_test  = Yn[test_idx]

print("Train/Val/Test sizes:", X_train.shape[0], X_val.shape[0], X_test.shape[0])

# Convert to torch
device = DEVICE
X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32, device=device)
X_val_t   = torch.tensor(X_val, dtype=torch.float32, device=device)
Y_val_t   = torch.tensor(Y_val, dtype=torch.float32, device=device)
X_all_t   = torch.tensor(Xn, dtype=torch.float32, device=device)  # for LBFGS final refine / full predictions

Train/Val/Test sizes: 826 91 91


In [4]:
def pde_residual_Cf(model, xb):
    """
    Physics residual for Grad-4Cf energy equation
    xb = [Epsilon, N, Eta] (normalized)
    """

    xb.requires_grad_(True)

    # Network output: theta(eta)
    theta = model(xb)

    # First derivative dθ/dη
    grad_theta = torch.autograd.grad(
        theta, xb,
        grad_outputs=torch.ones_like(theta),
        retain_graph=True,
        create_graph=True
    )[0]

    theta_eta = grad_theta[:, 2:3]

    # Second derivative d²θ/dη²
    grad2_theta = torch.autograd.grad(
        theta_eta, xb,
        grad_outputs=torch.ones_like(theta_eta),
        retain_graph=True,
        create_graph=True
    )[0]

    theta_eta2 = grad2_theta[:, 2:3]

    # Parameters
    epsilon = xb[:, 0:1]
    N  = xb[:, 1:2]

    # ---- Coefficients (from your notes, reduced form) ----
    a = 1.0 + epsilon                     # convection term
    b = N + epsilon**2                    # magnetic + buoyancy coupling

    # Energy equation residual
    residual = theta_eta2 + a * theta_eta + b * theta

    return residual


In [5]:
def loss_function(model, data_loss, bc_loss, x_pde, t_pde):
    loss = data_loss + bc_loss

    if USE_PDE:
        f_pde = pde_residual_Cf(model, x_pde, t_pde)
        pde_loss = torch.mean(f_pde**2)
        loss += pde_loss

    return loss


In [6]:
'''def physics_loss(model, xb):
    r = pde_residual_Cf(model, xb)
    return torch.mean(r**2) '''"Extract Data.ipynb"

def physics_loss(model, xb):
    r = pde_residual_Cf(model, xb)
    if torch.isnan(r).any():
        return torch.tensor(0.0, device=xb.device)
    return torch.mean(r**2)


In [7]:
def wall_bc_loss(model, xb):
    xb.requires_grad_(True)
    pred = model(xb)

    grads = torch.autograd.grad(
        pred, xb,
        grad_outputs=torch.ones_like(pred),
        create_graph=True
    )[0]

    dCf_dEta = grads[:, 2:3]

    # η ≈ 0 region (normalized)
    mask = xb[:, 2:3] < -1.5

    # 🚨 SAFETY CHECK
    if mask.sum() == 0:
        return torch.tensor(0.0, device=xb.device)

    return torch.mean(dCf_dEta[mask]**2)


In [8]:
def bc_loss(model, xb):
    """
    Boundary conditions for energy equation:
    theta(0) = 1
    theta(inf) = 0
    """

    eta = xb[:, 2:3]
    theta = model(xb)

    # ---- wall: eta = 0 ----
    # normalized eta ≈ 0 → use a small band
    mask_wall = torch.abs(eta) < 0.05
    if mask_wall.any():
        bc_wall = torch.mean((theta[mask_wall] - 1.0)**2)
    else:
        bc_wall = torch.tensor(0.0, device=xb.device)

    # ---- free stream: eta → ∞ ----
    mask_inf = eta > 1.5
    if mask_inf.any():
        bc_inf = torch.mean(theta[mask_inf]**2)
    else:
        bc_inf = torch.tensor(0.0, device=xb.device)

    return bc_wall + bc_inf


In [9]:
steps_per_epoch = max(
    1, int(np.ceil(X_train.shape[0] / float(BATCH_SIZE)))
)

total_steps = N_EPOCHS * steps_per_epoch


In [10]:
# ---------------- Model definition ----------------
class FGLongNet_3hidden(nn.Module):
    def __init__(self, in_dim=3, hidden=64, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU()
        )
        self.head = nn.Linear(hidden, out_dim)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        return self.head(self.net(x))


# 🔴 THIS LINE IS CRITICAL
model = FGLongNet_3hidden(in_dim=3, hidden=64, out_dim=1).to(device)


In [11]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-6)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=ONECYCLE_MAX_LR,
    total_steps=total_steps,
    pct_start=0.1,
    div_factor=10.0
)

mse = nn.MSELoss()


In [ ]:
# ---------------- Training loop ----------------
best_val = 1e12; patience = 0
history = {'train':[], 'val':[]}
t0 = time.time()
for ep in range(1, N_EPOCHS+1):
    model.train()
    perm = np.random.permutation(X_train.shape[0])
    losses=[]
    for i in range(0, X_train.shape[0], BATCH_SIZE):
        idxb = perm[i:i+BATCH_SIZE]
        #xb = torch.tensor(X_train[idxb], dtype=torch.float32, device=device)
        xb = torch.tensor(X_train[idxb], dtype=torch.float32, device=device, requires_grad=True)

        yb = torch.tensor(Y_train[idxb], dtype=torch.float32, device=device)
        optimizer.zero_grad()
        pred = model(xb)
        
        loss_data = mse(pred, yb)
        wall_bc = wall_bc_loss(model, xb)

        if USE_PDE:
            loss_pde = physics_loss(model, xb)
            loss_bc  = bc_loss(model, xb)
            loss_wall = wall_bc_loss(model, xb)
            loss = loss_data + 0.00001*loss_pde + 0.00001*(loss_bc + loss_wall)


        else:
            loss = loss_data    

        if torch.isnan(loss):
            print("NaN loss detected — skipping batch")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        try:
            scheduler.step()
        except Exception:
            pass
        losses.append(loss.item())
    train_loss = float(np.mean(losses))
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t); val_loss = float(mse(val_pred, Y_val_t).item())
    history['train'].append(train_loss); history['val'].append(val_loss)
    if ep % PRINT_EVERY == 0 or ep==1:
        print(f"Epoch {ep}/{N_EPOCHS} train={train_loss:.3e} val={val_loss:.3e} lr={optimizer.param_groups[0]['lr']:.2e}")
    if val_loss < best_val:
        best_val = val_loss; patience = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR,"best_long.pt"))
    else:
        patience += 1
        if patience > EARLY_STOPPING_PATIENCE:
            print("Early stopping at epoch", ep); break

t1 = time.time()
print("Training done in {:.1f}s best_val={:.4e}".format(t1-t0, best_val))
model.load_state_dict(torch.load(os.path.join(OUT_DIR,"best_long.pt"), map_location=device))
model.eval()

Epoch 1/12000 train=1.023e+00 val=1.021e+00 lr=8.00e-04
Epoch 100/12000 train=9.319e-01 val=1.047e+00 lr=9.23e-04
Epoch 200/12000 train=9.307e-01 val=1.030e+00 lr=1.28e-03
Epoch 300/12000 train=6.104e-01 val=6.876e-01 lr=1.85e-03
Epoch 400/12000 train=5.928e-01 val=6.553e-01 lr=2.60e-03
Epoch 500/12000 train=5.761e-01 val=5.789e-01 lr=3.47e-03
Epoch 600/12000 train=5.804e-01 val=6.357e-01 lr=4.40e-03
Epoch 700/12000 train=5.586e-01 val=6.363e-01 lr=5.33e-03
Epoch 800/12000 train=5.616e-01 val=5.567e-01 lr=6.20e-03
Epoch 900/12000 train=5.687e-01 val=5.776e-01 lr=6.95e-03
Epoch 1000/12000 train=5.941e-01 val=6.077e-01 lr=7.52e-03
Epoch 1100/12000 train=5.793e-01 val=5.822e-01 lr=7.88e-03
Epoch 1200/12000 train=5.497e-01 val=5.554e-01 lr=8.00e-03
Epoch 1300/12000 train=5.485e-01 val=5.368e-01 lr=8.00e-03
Epoch 1400/12000 train=5.817e-01 val=5.728e-01 lr=7.99e-03
Epoch 1500/12000 train=5.906e-01 val=4.993e-01 lr=7.98e-03
Epoch 1600/12000 train=5.891e-01 val=6.183e-01 lr=7.97e-03
Epoch 170

In [ ]:
# LBFGS refine (optional)
print("LBFGS refine...")
try:
    lbfgs = optim.LBFGS(model.parameters(), max_iter=LBFGS_ITERS)
    def closure():
        lbfgs.zero_grad()
        pred = model(X_all_t)
        loss = mse(pred, torch.tensor(Yn, dtype=torch.float32, device=device))
        loss.backward()
        return loss
    lbfgs.step(closure)
except Exception as e:
    print("LBFGS skipped/failed:", e)

# ---------------- Predict back to original scale ----------------
with torch.no_grad():
    Ypred_n = model(X_all_t).cpu().numpy()
Ypred = Ypred_n * Y_std + Y_mean   # invert normalization

# attach predictions to df and save
df['p_Cfval'] = Ypred[:,0]

out_csv = os.path.join(OUT_DIR, "Cf_long_preds.csv")
df.to_csv(out_csv, index=False)
print("Saved long-format predictions to:", out_csv)

# Compute RMSE overall and per combo and per k
df['err_Cf2'] = (df['p_Cfval'] - df['Cf_val'])**2

rmse_overall = np.sqrt(df[['err_Cf2']].mean(axis=0))
print("Overall RMSE (Cf):", rmse_overall.tolist())

# per k RMSE
rmse_k = df.groupby('k').apply(lambda g: np.sqrt(np.mean((g['p_Cfval']-g['Cf_val'])**2)))

rmse_df = pd.DataFrame({'k': rmse_k.index, 'rmse_Cf': rmse_k.values})
rmse_df.to_csv(os.path.join(OUT_DIR,"rmse_per_k.csv"), index=False)
print("Saved rmse_per_k.csv")

In [ ]:
# ---------------- plots: reconstruct curves per k (Epsilon,N) ----------------
for k in sorted(df['k'].unique()):
    sub = df[df['k']==k].sort_values('Eta')
    # use Cf_map (or G_map) to recover Epsilon,N for plotting
    epsilon, n = Cf_map[int(k)]
    plt.figure(figsize=(8,6))
    plt.plot(sub['Eta'], sub['Cf_val'], '-', label='Cf true')
    plt.plot(sub['Eta'], sub['p_Cfval'], '--', label='Cf pred')

    plt.title(f"k={int(k)} Epsilon={epsilon} N={n}")
    plt.legend()
    plt.xlabel("Eta")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"curve_k{int(k)}_Epsilon{epsilon}_N{n}.png"))
    plt.close()

print("Saved curve plots and parity/rmse info in", OUT_DIR)


In [ ]:
#Plot the training history
plt.figure(figsize=(8,6))
plt.plot(history['train'], label='Train Loss')
plt.plot(history['val'], label='Val Loss')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go
fig = go.Figure()
for k in range(1,5):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Cf_val'], mode='lines', name=f'Cf true Epsilon={epsilon} N={n}'))

fig.update_layout(title='Cf value vs Eta for different Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "Cf_values_plotly.html"))
fig.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go
fig = go.Figure()
for k in range(1,5):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]

    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Cfval'], mode='lines', name=f'Cf pred Epsilon={epsilon} N={n}'))

fig.update_layout(title='Cf values vs Eta for different Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "Cf_Pred_values_plotly.html"))
fig.show()

In [ ]:
#PLOT PLOTLY GRAPH FOR aCTUAL VALUES F1 to F12
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()

for k in range(1,5):
    sub = df[df['k']==k].sort_values('Eta')
    epsilon, n = Cf_map[int(k)]

    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['Cf_val'], mode='lines', name=f'Cf true Epsilon={epsilon} N={n}'))
    fig.add_trace(go.Scatter(x=sub['Eta'], y=sub['p_Cfval'], mode='markers', name=f'Cf pred Epsilon={epsilon} N={n}'))

fig.update_layout(title='Cf values vs Eta for different Epsilon, N', xaxis_title='Eta', yaxis_title='Values')
fig.write_html(os.path.join(OUT_DIR, "Cf_Vs_Epsilon_N_pred_values_plotly.html"))
fig.show()